In [38]:
import pandas as pd

df = pd.read_csv("processed_data.csv", engine='python', on_bad_lines='skip')
print(df.shape)
df.head()

(98978, 10)


,customer_id,product_id,quantity,price,transaction_date,payment_method,store_location,product_category,discount,total_amount
0,109318,C,7,80.079844,2023-12-26 12:32:00,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764
1,993229,C,4,75.195229,2023-08-05 00:00:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546
2,579675,A,8,31.528816,2024-03-11 18:51:00,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651
3,799826,D,5,98.880218,2023-10-27 22:00:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769
4,121413,A,7,93.188512,2023-12-22 11:38:00,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484


In [39]:
# Numerical summary
display(df.describe())

# Categorical summary
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())

,customer_id,quantity,price,discount,total_amount
count,98978.000000,98978.000000,98978.000000,98978.000000,98978.000000
mean,500509.983946,4.969135,54.658472,10.069519,242.668713
std,288437.456041,2.562306,25.786870,5.774971,176.797303
min,14.000000,1.000000,10.000430,0.000046,8.274825
25%,250715.250000,3.000000,32.313251,5.064056,94.286872
50%,499774.000000,5.000000,54.634706,10.104196,197.609865
75%,751136.750000,7.000000,76.846096,15.069394,356.165105
max,999997.000000,9.000000,99.999045,19.999585,751.333384



Column: product_id
product_id
C    24953
D    24785
B    24722
A    24518
Name: count, dtype: int64

Column: transaction_date
transaction_date
2024-04-03 23:55:00    5
2023-11-18 17:02:00    4
2024-03-29 13:03:00    4
2023-08-19 14:13:00    4
2023-09-04 11:20:00    4
                      ..
2023-06-24 07:16:00    1
2024-02-01 14:36:00    1
2023-07-27 12:35:00    1
2023-05-24 15:47:00    1
2024-03-01 13:17:00    1
Name: count, Length: 90167, dtype: int64

Column: payment_method
payment_method
Credit Card    24798
PayPal         24793
Cash           24751
Debit Card     24636
Name: count, dtype: int64

Column: store_location
store_location
8046 Hull Drive\nPaulstad, GU 87218                     1
176 Andrew Cliffs\nBaileyfort, HI 93354                 1
17448 Flores Springs Suite 225\nWhitetown, MO 34197     1
828 Andrews Plaza\nNew Jenniferhaven, VI 43622          1
69742 Hall Road\nWest Johnborough, MD 13284             1
                                                       ..
USNV

In [40]:
## 4️⃣ Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
customer_id         0
product_id          0
quantity            0
price               0
transaction_date    0
payment_method      0
store_location      0
product_category    0
discount            0
total_amount        0
dtype: int64


In [41]:
df = df.dropna()
print("New dataset shape:", df.shape)

New dataset shape: (98978, 10)


In [42]:

df = pd.read_csv("processed_data.csv")
df['transaction_date'] = pd.to_datetime(df['transaction_date'], dayfirst=True)

/tmp/ipykernel_1255/1514860889.py:2: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['transaction_date'] = pd.to_datetime(df['transaction_date'], dayfirst=True)


In [43]:
import datetime as dt

# Define snapshot date (today or one day after last transaction)
snapshot_date = df['transaction_date'].max() + pd.Timedelta(days=1)

# Aggregate by customer
rfm = df.groupby('customer_id').agg({
    'transaction_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'product_id': 'count',                                          # Frequency
    'total_amount': 'sum'                                           # Monetary
}).reset_index()

rfm.rename(columns={
    'transaction_date': 'Recency',
    'product_id': 'Frequency',
    'total_amount': 'Monetary'
}, inplace=True)

rfm.head()

,customer_id,Recency,Frequency,Monetary
0,14,267,1,256.232791
1,42,346,1,502.656523
2,49,329,1,21.399047
3,59,28,2,249.492696
4,65,316,1,548.006625


In [44]:
# Recency: lower = better
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])

# Frequency & Monetary: rank, then scale 1-4
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1,2,3,4])

In [45]:
# Combine to get RFM Segment and total score
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score','F_Score','M_Score']].sum(axis=1).astype(int)

# Preview
rfm.head()

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score
0,14,267,1,256.232791,2,1,3,213,6
1,42,346,1,502.656523,1,1,4,114,6
2,49,329,1,21.399047,1,1,1,111,3
3,59,28,2,249.492696,4,4,3,443,11
4,65,316,1,548.006625,1,1,4,114,6


In [46]:
rfm.to_csv("rfm_table.csv", index=False)